# Feature Engineering Pipeline - Titanic Dataset
과제 제출용 전체 실험 노트북입니다.

## 1. 데이터 로드 및 기본 구조 확인

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.data_loader import load_titanic
from src.feature_engineering import add_titanic_features
from src.eda import run_eda
from src.modeling import run_experiments, run_grid_search_for_best
from src.visualize_results import plot_metrics

import pandas as pd

df = load_titanic("../data/titanic.csv")
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe(include="all")

## 2. EDA - 결측치, 분포, 이상치, 상관관계, 타겟 분포

In [ ]:
missing = (df.isna().mean() * 100).sort_values(ascending=False)
missing

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(df["Age"], kde=True)
plt.title("Age Distribution")
plt.show()

In [ ]:
sns.boxplot(x=df["Fare"])
plt.title("Fare Boxplot")
plt.show()

In [ ]:
sns.heatmap(df.select_dtypes(include="number").corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
sns.countplot(x="Survived", data=df)
plt.title("Target Distribution")
plt.show()

## 3. 파생 변수 생성

In [ ]:
df_fe = add_titanic_features(df)
df_fe[["FamilySize", "IsAlone", "FarePerPerson", "AgeGroup", "Title"]].head()

## 4. 실험 비교 - 결측치 처리, 인코딩, 스케일링, Feature Selection

In [ ]:
results = run_experiments(df_fe)
results

In [ ]:
plot_metrics(results, "../results/figures")
results.to_csv("../results/metrics/experiment_results.csv", index=False)

## 5. GridSearchCV 가산점 실험

In [ ]:
grid_result = run_grid_search_for_best(df_fe)
grid_result

## 6. 결론 작성 예시
- F1-score와 ROC-AUC가 가장 높은 실험 조합을 최종 모델로 선정한다.
- One-Hot Encoding은 명목형 범주 처리에 유리하지만 차원이 증가할 수 있다.
- Feature Selection은 불필요한 변수를 줄여 과적합 완화에 기여할 수 있으나, 항상 성능 향상을 보장하지는 않는다.
- 스케일링은 Logistic Regression처럼 거리/계수 기반 모델에서 더 큰 영향을 보인다.